In [30]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from itertools import combinations
from scipy.stats import shapiro, friedmanchisquare, wilcoxon

In [31]:
TIME_CONDITIONS  = ["standard_input", "voice_control", "plugin"]

def get_task_wide(task_num):
    task_df = time_df[time_df["task"] == task_num]
    wide = task_df.pivot(index="participant", columns="input_mode", values="duration")
    wide = wide[TIME_CONDITIONS]
    wide = wide.dropna()
    wide = wide.reindex(sorted(wide.index, key=lambda p: int(p[1:])))
    return wide

def print_shapiro_result(condition_name, durations):
    stat, p = shapiro(durations)
    if p > 0.05:
        verdict = "normal"
    else:
        verdict = "NOT normal"
    print(f"  {condition_name}: W = {stat:.3f}, p = {p:.4f} -> {verdict}")

time_df = pd.read_csv("data/all_participants_summary.csv")

In [32]:
get_task_wide(1)

input_mode,standard_input,voice_control,plugin
participant,,,
V1,0.982,17.858,3.554
V2,0.857,11.256,3.091
V3,1.107,16.413,3.516
V4,0.739,16.001,3.399
V5,1.056,17.906,7.028
V6,1.480,15.660,5.475
V7,0.728,14.334,4.430
V8,0.814,11.719,5.794
V9,0.871,14.016,6.039


In [33]:
get_task_wide(2)

input_mode,standard_input,voice_control,plugin
participant,,,
V1,1.566,28.003,8.003
V2,2.619,40.390,1.967
V3,1.809,17.409,2.506
V4,3.409,84.019,5.753
V5,1.007,32.793,4.182
V6,3.406,36.569,4.736
V7,2.146,48.972,3.742
V8,2.555,84.374,5.664
V9,1.119,74.732,5.888


In [34]:
get_task_wide(3)

input_mode,standard_input,voice_control,plugin
participant,,,
V1,4.047,63.339,5.672
V2,3.513,93.714,2.752
V4,3.840,17.849,7.085
V5,3.183,56.424,4.267
V6,3.548,40.312,4.420
V7,3.854,40.243,5.262
V8,4.358,81.235,5.190
V9,2.308,41.768,7.081
V10,3.457,29.541,4.723


In [35]:
get_task_wide(4)

input_mode,standard_input,voice_control,plugin
participant,,,
V1,3.720,56.195,29.788
V3,4.112,62.824,9.391
V9,2.944,59.141,10.402
V11,5.278,50.758,27.268
V12,4.238,48.425,104.631
V14,4.937,93.192,108.078
V15,5.747,55.162,17.870


In [36]:
get_task_wide(5)

input_mode,standard_input,voice_control,plugin
participant,,,
V1,3.337,40.758,146.760
V2,2.947,46.107,36.585
V3,3.404,52.603,35.106
V5,3.116,48.463,62.650
V6,4.719,52.641,59.845
V7,3.085,37.500,49.009
V8,3.466,71.756,42.856
V9,2.659,40.004,22.327
V10,4.372,52.333,100.815


In [37]:
print("Shapiro-Wilk normality test (p > 0.05 = normal)")

for task_num in range(1, 6):
    wide = get_task_wide(task_num)
    print(f"\nTask {task_num} (n={len(wide)})")
    for condition in TIME_CONDITIONS:
        print_shapiro_result(condition, wide[condition])

Shapiro-Wilk normality test (p > 0.05 = normal)

Task 1 (n=15)
  standard_input: W = 0.881, p = 0.0490 -> NOT normal
  voice_control: W = 0.935, p = 0.3207 -> normal
  plugin: W = 0.972, p = 0.8866 -> normal

Task 2 (n=14)
  standard_input: W = 0.952, p = 0.5849 -> normal
  voice_control: W = 0.931, p = 0.3155 -> normal
  plugin: W = 0.968, p = 0.8448 -> normal

Task 3 (n=14)
  standard_input: W = 0.954, p = 0.6247 -> normal
  voice_control: W = 0.835, p = 0.0142 -> NOT normal
  plugin: W = 0.947, p = 0.5094 -> normal

Task 4 (n=7)
  standard_input: W = 0.981, p = 0.9623 -> normal
  voice_control: W = 0.757, p = 0.0148 -> NOT normal
  plugin: W = 0.746, p = 0.0116 -> NOT normal

Task 5 (n=14)
  standard_input: W = 0.926, p = 0.2643 -> normal
  voice_control: W = 0.875, p = 0.0492 -> NOT normal
  plugin: W = 0.837, p = 0.0147 -> NOT normal


In [38]:
print("Friedman test")
for task_num in range(1, 6):
    wide = get_task_wide(task_num)
    stat, p = friedmanchisquare(wide["standard_input"], wide["voice_control"], wide["plugin"])
    print(f"  Task {task_num} (n={len(wide)}): chi2 = {stat:.3f}, p = {p:.7f}")

Friedman test
  Task 1 (n=15): chi2 = 30.000, p = 0.0000003
  Task 2 (n=14): chi2 = 26.143, p = 0.0000021
  Task 3 (n=14): chi2 = 26.143, p = 0.0000021
  Task 4 (n=7): chi2 = 11.143, p = 0.0038050
  Task 5 (n=14): chi2 = 21.143, p = 0.0000256


In [39]:
ALPHA = 0.0167  # Bonferroni-corrected threshold for 3 pairwise comparisons

# Pairwise Wilcoxon signed-rank
time_rows = []
for task_num in range(1, 6):
    wide = get_task_wide(task_num)
    for a, b in combinations(TIME_CONDITIONS, 2):
        stat, p = wilcoxon(wide[a], wide[b])
        time_rows.append({
            "task": task_num,
            "n": len(wide),
            "comparison": f"{a} vs {b}",
            "W": stat,
            "p": round(p, 5),
            "sig": p < ALPHA,
        })

time_results = pd.DataFrame(time_rows)
time_results

,task,n,comparison,W,p,sig
0,1,15,standard_input vs voice_control,0.0,0.00006,True
1,1,15,standard_input vs plugin,0.0,0.00006,True
2,1,15,voice_control vs plugin,0.0,0.00006,True
3,2,14,standard_input vs voice_control,0.0,0.00012,True
4,2,14,standard_input vs plugin,1.0,0.00024,True
5,2,14,voice_control vs plugin,0.0,0.00012,True
6,3,14,standard_input vs voice_control,0.0,0.00012,True
7,3,14,standard_input vs plugin,1.0,0.00024,True
8,3,14,voice_control vs plugin,0.0,0.00012,True
9,4,7,standard_input vs voice_control,0.0,0.01562,True
